# Task 2 - PySpark Assignment Questions

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

# starting spark session
spark = SparkSession.builder.appName("Task2-Week6").getOrCreate()

In [0]:
# loading the ecommerce dataset to use for code questions
df = spark.read.parquet("/Volumes/workspace/default/week-6/ecommerce.parquet")
df.show(5)

+--------+---------------+--------------------+----------------+-----------------+--------+----------+----------------+------------+----------+--------------+---------+----------------+------------+------+
|order_id|  customer_name|               email|product_category|     product_name|quantity|unit_price|discount_percent|total_amount|order_date|          city|  country|  payment_method|order_status|rating|
+--------+---------------+--------------------+----------------+-----------------+--------+----------+----------------+------------+----------+--------------+---------+----------------+------------+------+
|ORD00001|      Wei Reddy|                NULL|  Home & Kitchen|     Coffee Maker|     2.0|    339.97|             0.0|      679.94|2024-08-27|         Tokyo|    Japan|     Credit Card|   Delivered|   5.0|
|ORD00002|     Ava Sharma|ava.sharma734@exa...|          Beauty|          Perfume|     8.0|    296.69|             0.0|     2373.52|2023-06-13|         Kyoto|    Japan|        

## Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

**Driver** - this is basically the brain of the spark application. its the main process that runs on one machine and it does all the planning work. it takes our code, figures out what needs to be done, builds a DAG (execution plan), and then tells the executors what tasks to run. it also keeps track of all the results and handles the SparkContext/SparkSession.

**Cluster Manager** - this is like a resource manager that sits between the driver and the actual worker machines. its job is to allocate resources (cpu, memory) across the cluster. so when the driver says "hey i need 4 executors with 8gb ram each", the cluster manager handles that. examples are YARN, Mesos, Kubernetes, or Sparks own standalone manager.

**Executor** - these are the actual worker processes that run on the cluster nodes and do the real computation. each executor runs the tasks assigned by the driver and stores data in memory or disk. one worker node can have multiple executors. they also report the status of tasks back to the driver so it knows whats going on.

## Q2: How does Spark's Lazy Evaluation strategy improve performance when chain-processing large datasets?

So in spark when we write transformations like filter, select, withColumn etc, they dont execute right away. spark just records them and builds a plan internally. it only actually runs everything when we call an action like show(), count(), collect() etc.

This is useful because spark gets to see the entire chain of operations before executing anything. so it can optimize things - like if i do a filter after a select, spark might push the filter before the select so it processes fewer rows early on. it can also combine multiple operations together so it doesnt have to read the data multiple times from disk.

basically if spark ran each transformation immediately, it would be really slow because itd have to materialize intermediate results at every step. with lazy evaluation it waits, looks at the full picture, optimizes the plan through the catalyst optimizer, and then runs everything in one efficient go. this saves a lot of unnecessary I/O and computation especially with big datasets.

## Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

In [0]:
# reading csv with header and inferschema
csv_df=spark.read.csv("/Volumes/workspace/default/week-6/ecommerce_orders_csv", header=True,inferSchema=True)
csv_df.show(5)

+--------+---------------+--------------------+----------------+-----------------+--------+----------+----------------+------------+----------+--------------+---------+----------------+------------+------+------------------+
|order_id|  customer_name|               email|product_category|     product_name|quantity|unit_price|discount_percent|total_amount|order_date|          city|  country|  payment_method|order_status|rating|    discount_price|
+--------+---------------+--------------------+----------------+-----------------+--------+----------+----------------+------------+----------+--------------+---------+----------------+------------+------+------------------+
|ORD00001|      Wei Reddy|                NULL|  Home & Kitchen|     Coffee Maker|       2|    339.97|             0.0|      679.94|2024-08-27|         Tokyo|    Japan|     Credit Card|   Delivered|   5.0|           611.946|
|ORD00002|     Ava Sharma|ava.sharma734@exa...|          Beauty|          Perfume|       8|    296.6

header=True tells spark that the first row has column names and not actual data. inferSchema=True makes spark scan through the data and figure out the datatypes automatically (like int, string, double etc) instead of treating everything as strings.

the generic syntax for any csv would be:
```python
csv_df = spark.read.csv("data/source.csv", header=True, inferSchema=True)
```

## Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

CSV is a row-based format - it stores data row by row, like how we see it in a spreadsheet. so if theres a row with 10 columns, all 10 values are stored together one after another and then the next row starts.

Parquet on the other hand is columnar - it stores all values of one column together, then all values of the next column and so on. 

Why this matters for performance:
- if i only need 2 columns out of 15, with CSV spark still has to read through all 15 columns for every row and discard the rest. with parquet it can directly jump to just those 2 columns and skip the others entirely. this is called column pruning.
- parquet compresses data way better because similar values in a column compress more efficiently than mixed data types in a row
- parquet also stores metadata like min/max values per column chunk, so spark can skip entire blocks of data if it knows the values wont match the filter. this is predicate pushdown.
- CSV files are larger on disk because theres no real compression by default and everything is plain text

So basically for analytics workloads where we typically query specific columns, parquet is much faster and more efficient.

## Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.

In [0]:
# selecting specific columns with a filter condition
# using actual dataset columns - order_id as product_id, unit_price as price, product_category as category
electronics_df = df.select("order_id", "unit_price").filter(col("product_category") == "Electronics")
electronics_df.show(5)

+--------+----------+
|order_id|unit_price|
+--------+----------+
|ORD00008|    376.68|
|ORD00015|     87.32|
|ORD00019|    136.27|
|ORD00021|     26.82|
|ORD00030|    101.25|
+--------+----------+
only showing top 5 rows


## Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.

In [0]:
# renaming column and casting datatype
# using our dataset - renaming customer_name to buyer_name and casting unit_price just as demo
revised_df = df.withColumnRenamed("customer_name", "buyer_name") \
               .withColumn("unit_price", col("unit_price").cast(DoubleType()))
revised_df.printSchema()
revised_df.show(5)

root
 |-- order_id: string (nullable = true)
 |-- buyer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- order_date: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- rating: double (nullable = true)

+--------+---------------+--------------------+----------------+-----------------+--------+----------+----------------+------------+----------+--------------+---------+----------------+------------+------+
|order_id|     buyer_name|               email|product_category|     product_name|quantity|unit_price|discount_percent|total_amount|order_date|          city|

## Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

Every time we apply a transformation in spark it doesnt just run it - it also records the "recipe" of how that RDD/DataFrame was created. this forms a chain called the lineage graph which is basically a DAG (Directed Acyclic Graph) of all the transformations from the original data source to the current state.

So if a worker node crashes and the data on that node (a partition) is lost, spark doesnt panic. it just looks at the lineage graph and knows exactly how to recreate that lost partition. it goes back to the source data and replays only the transformations needed for that specific lost partition.

the key thing is it doesnt have to recompute everything from scratch - just the partitions that were lost. and since the DAG records dependencies between partitions, spark can figure out the minimal amount of recomputation needed.

this is actually why spark doesnt need to replicate data across nodes like hadoop does. the lineage itself is the backup plan. though for really long chains of transformations we can use .persist() or .checkpoint() to save intermediate results so recomputation doesnt take too long.

## Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.

In [0]:
# filtering with multiple conditions using AND (&)
# using our dataset - order_status='Delivered' and total_amount > 1000 (since we dont have 'Completed' status)
filtered_orders = df.filter((col("order_status") == "Delivered") & (col("total_amount") > 1000))
filtered_orders.select("order_id", "customer_name", "order_status", "total_amount").show(5)

+--------+---------------+------------+------------+
|order_id|  customer_name|order_status|total_amount|
+--------+---------------+------------+------------+
|ORD00014|       Li Reddy|   Delivered|     1081.84|
|ORD00022|  Sophia Sharma|   Delivered|     1760.04|
|ORD00032|     Emma Rossi|   Delivered|     1262.19|
|ORD00053|  Vivaan Garcia|   Delivered|     3031.91|
|ORD00056|  Michael Rossi|   Delivered|     1038.46|
+--------+---------------+------------+------------+
only showing top 5 rows


## Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

Predicate pushdown means moving the filter condition (the predicate) as close to the data source as possible - basically pushing it "down" to the storage layer instead of loading everything first and then filtering.

Parquet files store metadata about each row group, things like the min value, max value and count for each column in that chunk. so when we write something like `df.filter(col("total_amount") > 1000)`, spark can check the parquet metadata before even reading the data. if a row group has max total_amount of 500, spark knows theres no point reading that entire chunk and just skips it.

This massively reduces how much data actually gets loaded into memory. instead of reading say 10GB of data and then throwing away 8GB after filtering, spark might only read 2GB that actually has matching rows. less disk I/O, less memory usage, faster queries.

This is one of the big reasons parquet is preferred over csv for analytics - csv cant do this at all because it has no column metadata, so spark has to read every single byte of the file regardless of the filter.

## Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [0]:
# adding a new column with 18% tax calculation
# using unit_price as base_price from our dataset
df_with_tax = df.withColumn("final_price", col("unit_price") * 1.18)
df_with_tax.select("order_id", "product_name", "unit_price", "final_price").show(5)

+--------+-----------------+----------+------------------+
|order_id|     product_name|unit_price|       final_price|
+--------+-----------------+----------+------------------+
|ORD00001|     Coffee Maker|    339.97|          401.1646|
|ORD00002|          Perfume|    296.69|          350.0942|
|ORD00003|     Storage Jars|    366.22|432.13960000000003|
|ORD00004|          Shampoo|    428.38|505.48839999999996|
|ORD00005|     Dumbbell Set|    138.62|          163.5716|
+--------+-----------------+----------+------------------+
only showing top 5 rows


## Q11: What is the difference between Transformations and Actions? Provide two examples of each.

**Transformations** are operations that create a new DataFrame/RDD from an existing one but dont actually execute anything immediately (lazy evaluation). they just add to the execution plan.

Examples:
- `filter()` - filters rows based on a condition, returns a new dataframe
- `select()` - picks specific columns, returns a new dataframe

**Actions** are operations that actually trigger the execution of all the queued transformations. they produce a result - either returning data to the driver or writing it somewhere.

Examples:
- `show()` - triggers computation and displays the rows on screen
- `count()` - triggers computation and returns the number of rows as a value

So basically transformations are lazy (they build the plan) and actions are eager (they run the plan). nothing happens until an action is called.

In [0]:
# quick demo showing transformations dont execute until action is called

# these are transformations - nothing happens yet
step1 = df.filter(col("product_category") == "Electronics")
step2 = step1.select("order_id", "product_name", "total_amount")

# this is the action - now everything above runs
step2.show(5)
print("count:", step2.count())

+--------+-----------------+------------+
|order_id|     product_name|total_amount|
+--------+-----------------+------------+
|ORD00008|Bluetooth Speaker|      2881.6|
|ORD00015|Bluetooth Speaker|      331.82|
|ORD00019|Bluetooth Speaker|     1226.43|
|ORD00021|Bluetooth Speaker|       134.1|
|ORD00030|          Earbuds|      759.38|
+--------+-----------------+------------+
only showing top 5 rows
count: 171


## Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

In [0]:
# read parquet -> filter nulls -> write csv (using our dataset as demo)
input_df = spark.read.parquet("/Volumes/workspace/default/week-6/ecommerce.parquet")
cleaned_df = input_df.filter(col("order_id").isNotNull())
cleaned_df.write.csv("/Volumes/workspace/default/week-6/ecommerce_cleaned", header=True, mode="overwrite")

In [0]:
# doing the same with our actual dataset as a demo
demo_df = spark.read.parquet("/Volumes/workspace/default/week-6/ecommerce.parquet")
# filtering out rows where email is null (since we dont have user_id column)
cleaned_demo = demo_df.filter(col("email").isNotNull())
print("before filter:", demo_df.count())
print("after removing nulls:", cleaned_demo.count())

before filter: 1000
after removing nulls: 953


## Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

The main difference is where the driver program runs.

**Client Mode:**
- the driver runs on the machine that submitted the spark job (like our laptop or an edge node)
- the executors still run on the cluster but the driver stays on the client side
- good for interactive work like notebooks, spark-shell, debugging etc because we can see the output directly
- downside is if the client machine disconnects or shuts down, the whole job dies
- also network can be a bottleneck since data has to travel between the cluster and the client machine

**Cluster Mode:**
- the driver runs on one of the worker nodes inside the cluster itself
- the client just submits the job and walks away, doesnt need to stay connected
- better for production jobs that run for hours because even if the client disconnects the job keeps running
- less network overhead since the driver is physically closer to the executors
- but harder to debug since we cant directly see the output, have to check logs

In databricks we mostly use cluster mode by default since the notebooks run on the cluster itself.

## Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

In [0]:
# filtering with OR condition (|)
# using our dataset columns - country and product_category since we dont have region/priority
or_filter = df.filter((col("country") == "Japan") | (col("product_category") == "Electronics"))
or_filter.select("order_id", "country", "product_category", "product_name").show(5)

+--------+---------+----------------+-----------------+
|order_id|  country|product_category|     product_name|
+--------+---------+----------------+-----------------+
|ORD00001|    Japan|  Home & Kitchen|     Coffee Maker|
|ORD00002|    Japan|          Beauty|          Perfume|
|ORD00007|    Japan|        Clothing|      Rain Jacket|
|ORD00008|   Canada|     Electronics|Bluetooth Speaker|
|ORD00013|    Japan|          Sports|   Cycling Helmet|
+--------+---------+----------------+-----------------+
only showing top 5 rows


## Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?

`.collect()` pulls the ENTIRE dataset from all the executor nodes back to the driver machine as a python list. so if the dataset is like 2TB, it tries to load all of that into the drivers memory which will almost certainly crash it with an OutOfMemoryError. the driver is just one machine with limited ram, its not meant to hold the entire dataset.

`.show(5)` on the other hand only fetches 5 rows to the driver. spark is smart enough to not process the entire dataset just to get 5 rows - it reads just enough partitions to fill those 5 rows and stops. so the memory usage is minimal.

so basically:
- `collect()` = bring everything to driver = dangerous on big data = can crash the driver
- `show(5)` = bring only 5 rows = safe = good for quick exploration

as a rule i usually use show() or take() for exploring data and only use collect() when im sure the data is small enough (like after a groupby that gives only a few rows).

In [0]:
# safe way to explore
df.show(5)
# df.collect()  # never do this on large data!

+--------+---------------+--------------------+----------------+-----------------+--------+----------+----------------+------------+----------+--------------+---------+----------------+------------+------+
|order_id|  customer_name|               email|product_category|     product_name|quantity|unit_price|discount_percent|total_amount|order_date|          city|  country|  payment_method|order_status|rating|
+--------+---------------+--------------------+----------------+-----------------+--------+----------+----------------+------------+----------+--------------+---------+----------------+------------+------+
|ORD00001|      Wei Reddy|                NULL|  Home & Kitchen|     Coffee Maker|     2.0|    339.97|             0.0|      679.94|2024-08-27|         Tokyo|    Japan|     Credit Card|   Delivered|   5.0|
|ORD00002|     Ava Sharma|ava.sharma734@exa...|          Beauty|          Perfume|     8.0|    296.69|             0.0|     2373.52|2023-06-13|         Kyoto|    Japan|        